In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def load_words(path="names.txt"):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

w = load_words()
words = w[:30]

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

In [4]:
block_size = 8

def build_dataset(words):
    X, Y = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for i in range(len(chs)-1):
            context = chs[:i+1]
            target = chs[i+1]

            context = context[-block_size:]
            context = ['.'] * (block_size - len(context)) + context

            X.append([stoi[c] for c in context])
            Y.append(stoi[target])

    return torch.tensor(X), torch.tensor(Y)

X, Y = build_dataset(words)

In [5]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        logits = self.linear(out)
        return logits
    
model = CharLSTM(vocab_size, embed_dim=16, hidden_dim=128)

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for step in range(2000):
    logits = model(X)

    logits_last = logits[:, -1, :]

    loss = torch.nn.functional.cross_entropy(logits_last, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(loss.item())

3.0711936950683594
0.5226898789405823
0.5151350498199463
0.5139309763908386
0.5135995745658875
0.5131964087486267
0.5138659477233887
0.5129866003990173
0.5129574537277222
0.5128971934318542


In [7]:
@torch.no_grad()
def generate_lstm(model):
    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context])
        logits = model(x)
        probs = torch.softmax(logits[:, -1, :], dim=1)

        ix = torch.multinomial(probs, 1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

for _ in range(20):
    print(generate_lstm(model))

mila
charlotte
madison
layla
luna
riley
abigail
avery
mia
sophia
mila
camila
emily
abigail
charlotte
sofia
charlotte
mila
amelia
abigail
